# Thực nghiệm 2.5 — Tái thiết kế trực quan hóa Golub et al. (1999)

**Phụ trách:** Thành viên 5 (TV5)  
**Mục tiêu:** Hiện đại hóa các hình trực quan kinh điển trong bài báo tiên phong của Golub et al. (Science, 1999):
- **Fig. 2**: Phân tích lân cận ALL/AML, đối chiếu nhãn thật và hoán vị nhãn bằng hình F6 của nhóm.
- **Fig. 4**: Khám phá lớp bằng SOM, đối chiếu thành phần cụm và minh họa trên không gian PCA.
- **Fig. 3A**: Phân bố Prediction Strength (PS) của 50 gen đặc trưng.
- **F04 (hình bổ sung của nhóm)**: Đánh giá Logistic Regression và phân tích lỗi trên tập kiểm tra độc lập (34 bệnh nhân), làm rõ mẫu nghi ngờ (mẫu 66) và các yếu tố gây nhiễu từ nguồn thu thập (loại mô BM/PB, bệnh viện).

Các hình xuất theo **Style Guide** (`docs/style_guide.md`) và quyết định kiến trúc **ADR-014 / ADR-061**.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

# Xac dinh thu muc goc du an
p = Path.cwd()
ROOT = None
while p != p.parent:
    if (p / 'src' / 'load.py').exists():
        ROOT = p
        break
    p = p.parent
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục dự án hoặc thư mục notebooks.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.load import load_golub
from src.evaluate import evaluate, _baseline_pipeline
from src.viz import save_figure
from src.viz_redesign import plot_fig4_som_redesign, plot_fig3a_prediction_strength, plot_fig4_error_analysis
from src.config import CURATED_DIR

print('Đã nạp các module dự án.')


## 1. Nạp dữ liệu tầng curated

`load_golub("curated")` đọc ma trận `X_log10.parquet` đã threshold và log10. Pipeline Logistic Regression fit các bước lọc và chọn gen trên tập train.

Phân cụm SOM dùng riêng ma trận `X_log10_filtered.parquet` gồm 3.571 probe, cùng đầu vào với mục 1.5.

In [ ]:
X_cu, samples = load_golub('curated')
X_som = pd.read_parquet(CURATED_DIR / 'X_log10_filtered.parquet')
print(f'Curated log10: {X_cu.shape[0]} mẫu × {X_cu.shape[1]} probe')
display(samples.head())

## 2. Fig. 2 — Phân tích lân cận

Dùng lại hình F6 từ [notebook 04_de_genes](04_de_genes.ipynb). Hai panel đếm các gen có điểm SNR vượt ngưỡng theo hai hướng ALL và AML. Các đường hoán vị nhãn cung cấp mức tham chiếu cho số lượng gen quan sát được.

![F6 — Phân tích lân cận](../results/figures/report/F6_neighborhood_report.png)

## 3. Fig. 4 — Khám phá lớp bằng SOM

Fig. 4A gốc có 24 ALL và 1 AML ở A1, 3 ALL và 10 AML ở A2. Hình nhóm đối chiếu thành phần hai cụm, chiếu các mẫu lên PCA và khảo sát nhóm con qua SOM bốn nút. Hai bản xuất giữ tên F02 trong danh mục hình của dự án.


In [ ]:
fig_som_report = plot_fig4_som_redesign(X_som, samples, mode='report')
save_figure(fig_som_report, 'F02_som_redesign', mode='report')
plt.show()

fig_som_slide = plot_fig4_som_redesign(X_som, samples, mode='slide')
save_figure(fig_som_slide, 'F02_som_redesign', mode='slide')
plt.close(fig_som_slide)
print('Da xuat F02_som_redesign cho ca report va slide.')

## 4. Tái thiết kế Fig 3A — Định lượng độ tin cậy dự đoán (Prediction Strength - Golub Fig. 3A)

- Nạp dữ liệu `results/metrics/golub_ps.csv` do TV1 cung cấp từ mô hình Golub Weighted Voting.
- Phân bố giá trị PS theo lớp ALL/AML trên tập huấn luyện (LOOCV, 38 mẫu) và tập kiểm tra độc lập (34 mẫu).
- Làm rõ vùng không chắc chắn (Uncertain, $PS < 0.30$) và đối chiếu trường hợp mẫu 66 và mẫu 67.

Trên tập kiểm tra, 9/10 mẫu PB được dự đoán đúng và mẫu 67 có nhãn `uncertain`. Mẫu 66 là AML, được dự đoán thành ALL với PS 0,30004.


In [ ]:
ps_df = pd.read_csv(ROOT / 'results' / 'metrics' / 'golub_ps.csv')
print('Tong so mau PS:', len(ps_df))

# Ve va xuat ban report
fig3a_rep = plot_fig3a_prediction_strength(ps_df, mode='report')
path3a_rep = save_figure(fig3a_rep, 'F03a_prediction_strength', mode='report')
plt.show()

# Ve va xuat ban slide
fig3a_sld = plot_fig3a_prediction_strength(ps_df, mode='slide')
path3a_sld = save_figure(fig3a_sld, 'F03a_prediction_strength', mode='slide')
plt.close(fig3a_sld)
print('Da xuat F03a thanh cong vao results/figures/')


## 5. F04 — Phân tích lỗi Logistic Regression trên tập kiểm tra độc lập

Dải điểm AML 0,3–0,7 là vùng tham chiếu trực quan. Ngưỡng phân loại là 0,5.

In [ ]:
pipe = _baseline_pipeline(k=50)
res_test = evaluate(pipe, X_cu, samples['class'], scheme='original_split')
preds_test = res_test['predictions']
errors = preds_test.loc[preds_test['y_pred'].ne(preds_test['y_true'])]
print(f'{len(errors)}/{len(preds_test)} mẫu phân loại sai')
display(errors)

fig4_report = plot_fig4_error_analysis(preds_test, samples, mode='report')
save_figure(fig4_report, 'F04_error_analysis', mode='report')
plt.show()

fig4_slide = plot_fig4_error_analysis(preds_test, samples, mode='slide')
save_figure(fig4_slide, 'F04_error_analysis', mode='slide')
plt.close(fig4_slide)
print('Da xuat F04_error_analysis cho ca report va slide.')

## 6. Kết luận

- **Fig. 2:** Hình F6 cho thấy tín hiệu biểu hiện liên hệ với nhãn ALL/AML trên tập train khi đối chiếu với nhãn hoán vị.
- **Fig. 3A:** LOOCV có 36/38 mẫu chắc chắn và đúng 36/36. Tập kiểm tra có 33/34 mẫu chắc chắn và đúng 32/33. Trong 10 mẫu PB, 9 mẫu đúng và mẫu 67 không chắc chắn.
- **Fig. 4:** SOM hai nút trong bài gốc có 34/38 mẫu cùng lớp với đa số cụm. Với đầu vào của mục 1.5, kết quả nhóm thay đổi theo seed và T-ALL chưa tạo cụm riêng.
- **F04 bổ sung:** Logistic Regression phân loại đúng 33/34 mẫu, gồm cả 10 mẫu PB. Mẫu 66 là ca phân loại sai.
